# Figures A4 & A5: Overall engine efficiency comparisons

In [ ]:
import random
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/landsat_sentinel_collocations_20260216.csv', comment='#')

In [ ]:
df = df[df["air_temperature_iagos_validity"] <= 0]
df = df[df["rhl_iagos_validity"] <= 0]
df = df[df["contrail_formation"].notna()]
df = df[df["efficiency_PS_IAGOS"].notna()]
df = df[df["efficiency_BADA4_IAGOS"].notna()]

df

In [ ]:

plt.rcParams.update({
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})

# Prepare flight phase function
def get_phase(vr):
    if abs(vr) < 100:
        return 'cruise'
    elif vr >= 100:
        return 'climb'
    else:
        return 'descent'

df['phase'] = df['vertical_rate'].apply(get_phase)

icao_codes = df['icao_code'].unique()
flight_phases = ['cruise', 'climb', 'descent']

# --- Color map for ICAO codes ---
colors = plt.cm.tab10.colors
color_map = {code: colors[i % len(colors)] for i, code in enumerate(icao_codes)}

# --- Marker map for flight phases ---
markers = ['o', '^', 's', 'D', 'v', 'P', 'X', '*']
marker_map = {code: markers[i % len(markers)] for i, code in enumerate(icao_codes)}

phase_marker_map = {
    'cruise': 'o',
    'climb': '^',
    'descent': 'v'
}

color_map = {
    "A333": '#0076c2',
    "A332": '#ec6842',
    "A343": "#009b77",
}


# --- Create figure with gridspec ---
fig = plt.figure(figsize=(6,6), dpi=300)
gs = fig.add_gridspec(4,4)
ax_main = fig.add_subplot(gs[1: , 0:3])
ax_xhist = fig.add_subplot(gs[0, 0:3], sharex=ax_main)
ax_yhist = fig.add_subplot(gs[1:, 3], sharey=ax_main)

# --- Scatter plot on main axes ---
for code in icao_codes:
    subset = df[df['icao_code'] == code]
    for phase in flight_phases:
        phase_subset = subset[subset['phase'] == phase]
        ax_main.scatter(
            phase_subset['efficiency_PS_IAGOS'],
            phase_subset['efficiency_BADA4_IAGOS'],
            alpha=0.6,
            color=color_map[code],        # <-- color = aircraft
            marker=phase_marker_map[phase],     # <-- marker = phase
            s=50,
            clip_on=False
        )

# 1:1 line
ax_main.plot([0,0.4],[0,0.4], color='grey', linestyle='--', linewidth=2)

# Axes limits, ticks, grid
ax_main.set_xlim(0,0.4)
ax_main.set_ylim(0,0.4)
ax_main.set_xticks(np.arange(0,0.5,0.1))
ax_main.set_yticks(np.arange(0,0.5,0.1))
ax_main.set_xlabel("Efficiency PS (-)")
ax_main.set_ylabel("Efficiency BADA4 (-)")
ax_main.grid(True, which='major')
ax_main.set_aspect('equal', adjustable='box')
ax_main.set_axisbelow(True)

# --- Marginal distributions (per aircraft ICAO) ---
for code in icao_codes:
    subset = df[df['icao_code'] == code]

    ax_xhist.hist(
        subset['efficiency_PS_IAGOS'],
        bins=np.arange(0, 0.45, 0.01),
        density=True,
        histtype='stepfilled',
        alpha=0.4,
        linewidth=1,
        edgecolor=color_map[code],
        color=color_map[code]
    )

    ax_yhist.hist(
        subset['efficiency_BADA4_IAGOS'],
        bins=np.arange(0, 0.45, 0.01),
        density=True,
        histtype='stepfilled',
        alpha=0.4,
        linewidth=1,
        orientation='horizontal',
        edgecolor=color_map[code],
        color=color_map[code]
    )


ax_xhist.axis('off')
ax_yhist.axis('off')

# --- Separate legends ---
handles_color = [
    plt.Line2D([0],[0],
               marker='o',
               linestyle='',
               color=color_map[code],
               markersize=8,
               label=code)
    for code in icao_codes
]

legend1 = ax_main.legend(
    handles=handles_color,
    title="Aircraft ICAO",
    loc="upper left"
)

handles_marker = [
    plt.Line2D([0],[0],
               marker=phase_marker_map[phase],
               linestyle='',
               color='k',
               markersize=8,
               label=phase.capitalize())
    for phase in flight_phases
]

legend2 = ax_main.legend(
    handles=handles_marker,
    title="Flight phase",
    loc="lower right"
)

ax_main.add_artist(legend1)

plt.tight_layout()
plt.savefig("../figures/figA4.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
# ===== FIGURE 2: Vertical Rate vs Efficiencies =====
color_PS = "#ffb81d"      # PS efficiency
color_BADA4 = "#1c1c73"  # BADA4 efficiency
hist_color = "#7f7f7f"   # marginal histograms

fig2 = plt.figure(figsize=(6, 6), dpi=300)
gs = fig2.add_gridspec(4, 4)

ax_main = fig2.add_subplot(gs[1:, 0:3])
ax_xhist = fig2.add_subplot(gs[0, 0:3], sharex=ax_main)
ax_yhist = fig2.add_subplot(gs[1:, 3], sharey=ax_main)

for phase, marker in phase_marker_map.items():
    subset = df[df['phase'] == phase]

    ax_main.scatter(
        subset['vertical_rate'],
        subset['efficiency_PS_IAGOS'],
        alpha=0.6,
        label=f"PS – {phase}",
        marker=marker,
        color=color_PS,
        clip_on=False,
        zorder=3,
        s=60
    )

for phase, marker in phase_marker_map.items():
    subset = df[df['phase'] == phase]

    ax_main.scatter(
        subset['vertical_rate'],
        subset['efficiency_BADA4_IAGOS'],
        alpha=0.6,
        label=f"BADA4 – {phase}",
        marker=marker,
        color=color_BADA4,
        clip_on=False,
        zorder=3,
        s=60
    )

handles_color = [
    plt.Line2D([0],[0], marker='o', color=color_PS, linestyle='', label='PS'),
    plt.Line2D([0],[0], marker='o', color=color_BADA4, linestyle='', label='BADA4')
]

handles_marker = [
    plt.Line2D([0],[0], marker=m, color='k', linestyle='', label=p.capitalize())
    for p, m in phase_marker_map.items()
]

legend1 = ax_main.legend(handles=handles_color, title="Model", loc="upper left")
legend2 = ax_main.legend(handles=handles_marker, title="Flight phase", loc="lower right")
ax_main.add_artist(legend1)

ax_main.axvline(0, color='grey', linewidth=2, linestyle="--", zorder=2)

ax_main.set_xlim(-3500, 3500)
ax_main.set_ylim(0, 0.5)

ax_main.set_xlabel("Vertical Rate (ft/min)")
ax_main.set_ylabel("Efficiency (-)")
ax_main.grid(True)

ax_xhist.hist(
    df['vertical_rate'],
    bins=np.arange(-3500, 3600, 100),
    density=True,
    color=hist_color,
    alpha=0.6,
    histtype="stepfilled",
    edgecolor=hist_color
)

ax_yhist.hist(
    df['efficiency_PS_IAGOS'],
    bins=np.arange(0, 0.55, 0.01),
    density=True,
    orientation='horizontal',
    color=color_PS,
    alpha=0.4,
    histtype="stepfilled",
    edgecolor=color_PS
)

ax_yhist.hist(
    df['efficiency_BADA4_IAGOS'],
    bins=np.arange(0, 0.55, 0.01),
    density=True,
    orientation='horizontal',
    color=color_BADA4,
    alpha=0.4,
    histtype="stepfilled",
    edgecolor=color_BADA4
)

ax_xhist.axis('off')
ax_yhist.axis('off')

plt.tight_layout()
plt.savefig("../figures/figA5.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

df1 = df[df["vertical_rate"] >= -100]
for icao in df["icao_code"].unique():
    df_icao = df1[df1["icao_code"] == icao]
    print(icao, len(df_icao))

    x = df_icao["efficiency_BADA4_IAGOS"].values
    y = df_icao["efficiency_PS_IAGOS"].values

    # Root mean square difference
    rmsd = np.sqrt(np.mean((x - y)**2))

    # R² coefficient of determination
    r2 = r2_score(x, y)

    # Slope constrained through origin
    a = np.sum(x * y) / np.sum(x**2)

    # Predictions with zero-intercept model
    y_pred = a * x

    # R² with intercept fixed at 0
    r2_origin = 1 - np.sum((y - y_pred)**2) / np.sum(y**2)

    from scipy.stats import pearsonr
    r, _ = pearsonr(x, y)

    print("BADA4-PS RMSD:", rmsd)
    print(f"BADA4-PS R²:", r2)
    print("BADA4-PS R² (through origin):", r2_origin)
    print("BADA4-PS Pearson r:", r)
    print()

In [ ]:
# Define flight phases
climb_mask   = df["vertical_rate"] > 100
cruise_mask  = df["vertical_rate"].abs() < 100
descent_mask = df["vertical_rate"] < -100

phases = {
    "climb": climb_mask,
    "cruise": cruise_mask,
    "descent": descent_mask
}

# Build results table
out = pd.DataFrame()

for apm in ["BADA4", "PS"]:
    for phase_name, mask in phases.items():

        # Filter to phase
        df_phase = df.loc[mask]

        # Compute mean IAGOS efficiency grouped by ICAO code
        means = df_phase.groupby("icao_code")[f"efficiency_{apm}_IAGOS"].mean()

        # Column name example: 'PS_climb'
        col_name = f"{apm}_{phase_name}"

        out[col_name] = means

# Sort rows by ICAO code
out = out.sort_index()
out